# Exploratory notebook — arXiv:2607.18311

Reproduces the paper's **calibration plot (Fig. 7)** and **deviation
histograms (Fig. 8)** from a trained checkpoint's predictions on the test set.

**Requires**: a checkpoint from `python train.py` (e.g. `outputs/best_model.pt`)
and the real dataset (`python data/download.py`). If you only have the
`data/toy/` fixture, this notebook will still run but the plots will be
near-meaningless on 1-2 test pairs -- that's expected.

In [ ]:
import sys
sys.path.insert(0, "../src")
import torch
import matplotlib.pyplot as plt

from spr_gnn.utils.config import Config, resolve_device
from spr_gnn.data.dataset import TreePairDataModule
from spr_gnn.models.siamese_gin import SiameseGINRegressor
from spr_gnn.evaluation.metrics import RegressionMetrics

CHECKPOINT = "../outputs/best_model.pt"  # update if yours lives elsewhere
CONFIG = "../configs/config.yaml"

config = Config.load(CONFIG)
device = resolve_device(config.get("hardware", "device", "cuda_if_available_else_cpu"))


In [ ]:
dm = TreePairDataModule(config)
dm.setup()
test_loader = dm.test_dataloader()

model = SiameseGINRegressor(
    num_species=dm.num_species,
    num_gin_layers=config.get("model", "num_gin_layers", 2),
).to(device)

try:
    ckpt = torch.load(CHECKPOINT, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded checkpoint from {CHECKPOINT}")
except FileNotFoundError:
    print(f"No checkpoint found at {CHECKPOINT} -- run train.py first. Using randomly-initialized weights for now.")


In [ ]:
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for tree_a, tree_b, target in test_loader:
        tree_a, tree_b = tree_a.to(device), tree_b.to(device)
        pred = model(tree_a, tree_b).cpu()
        all_preds.append(pred)
        all_targets.append(target)

preds = torch.cat(all_preds).numpy()
targets = torch.cat(all_targets).numpy()

metrics = RegressionMetrics()
result = metrics.compute(preds, targets)
print(result)


## Fig. 7 — Calibration plot

Model-predicted vs. true SPR distance, with the ideal $y=x$ line. The paper
notes predictions "track the diagonal but compress towards the mean, with
short distances overestimated and long distances underestimated" (Sec 5.3).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(targets, preds, alpha=0.6, label="Model predictions")
lims = [0, max(targets.max(), preds.max()) * 1.05 + 1e-6]
ax.plot(lims, lims, "r--", label="Perfect calibration (y = x)")
ax.set_xlabel("True SPR distance")
ax.set_ylabel("Model-predicted SPR distance")
ax.set_title("Model Calibration Plot")
ax.legend()
plt.tight_layout()
plt.show()


## Fig. 8 — Deviation histograms

Left: absolute deviation ($\delta_i = pred_i - real_i$). Right: normalized
deviation ($\delta_i / real_i$).

In [ ]:
hist_data = metrics.deviation_histograms(preds, targets)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(hist_data["absolute_deviation"], bins=20, color="#4C72B0")
axes[0].axvline(0, color="red", linestyle="--", label="Ideal Prediction (0)")
axes[0].set_title("Histogram of Absolute Deviation")
axes[0].set_xlabel("Absolute Difference (delta)")
axes[0].legend()

axes[1].hist(hist_data["normalized_deviation"], bins=20, color="#55A868")
axes[1].axvline(0, color="red", linestyle="--", label="Ideal Prediction (0)")
axes[1].set_title("Histogram of Normalized Deviation")
axes[1].set_xlabel("Relative Deviation Proportion")
axes[1].legend()

plt.tight_layout()
plt.show()


## Compare against the paper's numbers

See `paper_results` in `notebooks/reproduce_arxiv_2607_18311.ipynb`, or feed
your `results/eval_results.json` (from `evaluate.py`) into ArXivist's
Results Comparator (Stage 6) for a full comparison report.